# RealCause Lalonde HPO Benchmark: Foundation Models vs. Metalearners

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/layer6ai-labs/causalfm-survey/blob/main/notebooks/RealCause_with_hpo_benchmark.ipynb)

The production version of the RealCause Lalonde benchmark: 3 foundation models plus 6
metalearners, with FLAML-tuned nuisance models for the metalearners (`hpo=True`), on the
**arXiv v1 "first-10 realizations"** protocol CausalPFN's Table 1 reports. (CausalPFN's
current arXiv v2 uses all 100 realizations and reports different numbers — this notebook
targets v1 to stay comparable to the table below.)

| Cohort | PEHE (×10³) | ATE relative error |
|---|---|---|
| PSID | 13.98 ± 0.43 | 0.20 ± 0.03 |
| CPS | 8.83 ± 0.04 | 0.08 ± 0.02 |

Results are saved in raw dollars; only the printed summary table (§7) divides PEHE by 1,000
to compare directly against this table.

A full run is expensive — 6 metalearners × 2 propensity/outcome FLAML searches × ~3.5
nuisance fits each × 2 cohorts × 10 realizations is **420 FLAML searches at 900s each, ~105
CPU-hours** — so every cell below is written to survive being interrupted partway through:
exact dependency/vendor pinning (so a resumed run can't silently mix versions), and
atomic, per-task checkpointing with resume-on-rerun (§6).

## 1. Environment: exact dependency pins, verified before any import

Run top-to-bottom in a fresh kernel. Unlike the other notebooks in this repo (which only
pin the one or two packages known to conflict), this one pins every dependency to an exact,
tested version — worth the extra rigidity given how expensive a full run is: a version drift
discovered 80 hours in is a much worse time to find out than at cell 1.

On Colab, a mismatch is installed automatically; if a stale version was already imported
this session, you're told to restart rather than continuing on partially-updated packages.
Locally, this only *detects* problems and prints the exact `uv pip install` command to fix
them — this repo's `uv` venv has no `pip` module for the cell to install anything itself.

In [ ]:
import importlib.metadata as importlib_metadata
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # required for deterministic CUDA ops

# dist name -> exact tested version. FLAML/tabpfn/einops/tensorboard/networkx/tqdm are only
# needed for HPO/Do-PFN/CausalFM; pinned here too so the whole environment is reproducible.
PINNED_VERSIONS = {
    "torch": "2.9.1", "numpy": "2.2.6", "pandas": "2.3.3",
    "scikit-learn": "1.6.1", "econml": "0.17.0", "FLAML": "2.3.5",
    "causalpfn": "0.1.4", "networkx": "3.4.2", "tqdm": "4.68.3",
    "einops": "0.8.2", "tabpfn": "2.0.9", "tensorboard": "2.21.0",
    "scipy": "1.15.3", "xgboost": "2.1.4", "lightgbm": "4.7.0",
}
PIP_INSTALL_SPECS = [
    f"FLAML[automl]=={PINNED_VERSIONS['FLAML']}" if dist == "FLAML" else f"{dist}=={version}"
    for dist, version in PINNED_VERSIONS.items()
]
IMPORT_MODULE_NAMES = {  # dist name -> the name you'd `import`, where they differ
    "scikit-learn": "sklearn", "FLAML": "flaml",
}

def installed_version(dist):
    try:
        return importlib_metadata.version(dist)
    except importlib_metadata.PackageNotFoundError:
        return None

if "causal_bench" in sys.modules:
    raise RuntimeError("Restart the kernel: causal_bench was imported before this environment check ran.")

mismatched = {
    dist: (installed_version(dist), version)
    for dist, version in PINNED_VERSIONS.items()
    if installed_version(dist) != version
}

if IN_COLAB and mismatched:
    already_imported = [
        IMPORT_MODULE_NAMES.get(dist, dist.replace("-", "_"))
        for dist in mismatched
        if IMPORT_MODULE_NAMES.get(dist, dist.replace("-", "_")) in sys.modules
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *PIP_INSTALL_SPECS], check=True)
    if already_imported:
        raise RuntimeError(
            f"Pins installed, but these modules were already loaded with old versions: "
            f"{already_imported}. Restart the Colab runtime now, then rerun from the top."
        )
elif mismatched:
    fix_command = " ".join(repr(spec) for spec in PIP_INSTALL_SPECS)
    raise RuntimeError(
        f"Exact tested pins are required for this notebook; found mismatches: {mismatched}.\n"
        f"Run in a terminal: uv pip install {fix_command}\nThen restart the kernel."
    )

still_mismatched = {
    dist: (installed_version(dist), version)
    for dist, version in PINNED_VERSIONS.items()
    if installed_version(dist) != version
}
if still_mismatched:
    raise RuntimeError(f"Pin verification failed after install: {still_mismatched}")

print("All dependency pins verified before torch/causal_bench import.")

## 2. Project root and vendored model repos, pinned to exact commits

Do-PFN and CausalFM aren't on PyPI (see `CLAUDE.md`), so they're `git clone`d directly —
but for a run this expensive, "whatever the default branch currently has" isn't good
enough: it's pinned to the exact commit each was last verified against, checked out with
`git checkout --detach`. If a previously-cloned copy has local changes, this refuses to
overwrite them rather than silently discarding your work.

In [ ]:
from pathlib import Path

def run_command(command):
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f"Failed: {' '.join(command)}\n{result.stdout}\n{result.stderr}")
    return result.stdout.strip()

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "causal_bench").is_dir() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    return None

if IN_COLAB:
    PROJECT_ROOT = Path("/content/causalfm-survey")
    if PROJECT_ROOT.exists():
        run_command(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"])
    else:
        run_command(["git", "clone", "https://github.com/layer6ai-labs/causalfm-survey.git", str(PROJECT_ROOT)])
else:
    PROJECT_ROOT = find_project_root()
    if PROJECT_ROOT is None:
        raise RuntimeError("Run Jupyter from within this repository (or a subdirectory of it).")

# repo -> (clone URL, exact commit this notebook was last verified against)
VENDORED_REPOS = {
    "Do-PFN": ("https://github.com/jr2021/Do-PFN.git", "90d67433b43c4d52d752dc336070f525ff856e0b"),
    "CausalFM-toolkit": ("https://github.com/yccm/CausalFM-toolkit.git", "bb74ef729c70d274fe2bb422c3b0edff4754fa37"),
}

def pin_vendor_repo(name, url, sha):
    path = PROJECT_ROOT / "notebooks" / name
    if not path.exists():
        run_command(["git", "clone", url, str(path)])
    head = run_command(["git", "-C", str(path), "rev-parse", "HEAD"])
    if head != sha:
        dirty = run_command(["git", "-C", str(path), "status", "--porcelain", "--untracked-files=no"])
        if dirty:
            raise RuntimeError(f"Refusing to overwrite local changes in {name}:\n{dirty}")
        run_command(["git", "-C", str(path), "fetch", "origin", sha])
        run_command(["git", "-C", str(path), "checkout", "--detach", sha])
    got = run_command(["git", "-C", str(path), "rev-parse", "HEAD"])
    if got != sha:
        raise RuntimeError(f"{name}: checked out {got}, expected {sha}")
    return path.resolve(), got

DOPFN_DIR, dopfn_sha = pin_vendor_repo("Do-PFN", *VENDORED_REPOS["Do-PFN"])
CAUSALFM_DIR, causalfm_sha = pin_vendor_repo("CausalFM-toolkit", *VENDORED_REPOS["CausalFM-toolkit"])
VENDOR_SHAS = {"Do-PFN": dopfn_sha, "CausalFM-toolkit": causalfm_sha}
CAUSALFM_CHECKPOINT = CAUSALFM_DIR / "checkpoints/checkpoints_standard/best_model.pth"

print("Vendored repos pinned:", VENDOR_SHAS)

Pretrained checkpoints ship inside those repos rather than being downloaded separately —
check they're actually present (and not, say, an un-pulled Git LFS pointer) before the run
gets underway rather than failing confusingly mid-fit.

In [ ]:
REQUIRED_ARTIFACTS = [
    (CAUSALFM_CHECKPOINT, 10_000_000),
    (DOPFN_DIR / "artifacts/model_submitit_0ccc_id_171b69db_epoch_-1.cpkt", 10_000_000),
    (DOPFN_DIR / "artifacts/dopfn_model.pkl", 10_000_000),
    (DOPFN_DIR / "artifacts/dopfn_config.pkl", 100),
]
for path, minimum_bytes in REQUIRED_ARTIFACTS:
    if not path.is_file() or path.stat().st_size < minimum_bytes:
        raise RuntimeError(f"Missing or truncated model artifact: {path}")
    with path.open("rb") as f:
        header = f.read(128)
    if header.startswith(b"version https://git-lfs"):
        raise RuntimeError(f"{path} is a Git LFS pointer, not the real artifact -- run `git lfs pull`.")

# PROJECT_ROOT first (highest priority), then the two vendored repos.
for path in reversed((PROJECT_ROOT, DOPFN_DIR, CAUSALFM_DIR)):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print("All model artifacts present.")

## 3. Determinism, device, and the `causal_bench` model registry

Every task below is seeded individually (§6) so results are reproducible across resumed
runs, not just across a single unbroken one. `torch.use_deterministic_algorithms` is set
`warn_only=True` rather than strict: a handful of CUDA ops used here have no fully
deterministic kernel, and failing outright on those would block GPU runs entirely.

In [ ]:
import gc
import hashlib
import json
import random
import re
import time
import traceback
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

def env_flag(name, default):
    raw = os.environ.get(name, "1" if default else "0").strip().lower()
    if raw in {"1", "true", "yes", "on"}:
        return True
    if raw in {"0", "false", "no", "off"}:
        return False
    raise ValueError(f"Bad boolean environment value {name}={raw!r}")

REQUIRE_CUDA = env_flag("CAUSAL_BENCH_REQUIRE_CUDA", True)
FAIL_FAST = env_flag("CAUSAL_BENCH_FAIL_FAST", True)
CAUSALPFN_CAP_NEIGHBOURS = env_flag("CAUSAL_BENCH_CAUSALPFN_CAP_NUM_NEIGHBOURS", True)

_causalfm_batch_size = int(os.environ.get("CAUSAL_BENCH_CAUSALFM_QUERY_BATCH_SIZE", "0"))
if _causalfm_batch_size < 0:
    raise ValueError("CAUSAL_BENCH_CAUSALFM_QUERY_BATCH_SIZE must be 0 (native) or a positive integer")
CAUSALFM_QUERY_BATCH_SIZE = None if _causalfm_batch_size == 0 else _causalfm_batch_size

if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA required for the foundation models. Select a GPU runtime, or set "
        "CAUSAL_BENCH_REQUIRE_CUDA=0 to run an HPO-only CPU pass."
    )
device = "cuda" if torch.cuda.is_available() else "cpu"

BASE_SEED = 82718
random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASE_SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
warnings.filterwarnings("default")  # this is an audit run -- surface every warning, don't hide any

from causal_bench import (
    CausalPFNWrapper, DoPFNWrapper, CausalFMWrapper, HPOConfig,
    SLearnerWrapper, TLearnerWrapper, XLearnerWrapper, DebiasedMLWrapper, IPWWrapper, DRWrapper,
)

print("Device:", device, "| FAIL_FAST:", FAIL_FAST, "| CausalPFN cap_num_neighbours:", CAUSALPFN_CAP_NEIGHBOURS)

### Run configuration

Production defaults run all 9 models, all 10 realizations, both cohorts, with a 900s FLAML
budget per nuisance-model search, and require CUDA. Override any of these with environment
variables for a smaller/staged run:

| Variable | Effect |
|---|---|
| `CAUSAL_BENCH_N_REALIZATIONS=1` | Smoke-test with 1 realization instead of 10 (§5) |
| `CAUSAL_BENCH_HPO_TIME_BUDGET=1` | Cut the FLAML budget per search to `1`s for a fast smoke test |
| `CAUSAL_BENCH_MODEL_FILTER=IPW` | Only run the named model(s), comma-separated |
| `CAUSAL_BENCH_RUN_HPO=0` | Skip metalearners — a GPU-only foundation-model pass |
| `CAUSAL_BENCH_RUN_FOUNDATION=0`, `CAUSAL_BENCH_REQUIRE_CUDA=0` | Skip foundation models — a CPU-only HPO pass |
| `CAUSAL_BENCH_OUTPUT_DIR=/path` | Write results somewhere persistent (e.g. across Colab sessions) |
| `CAUSAL_BENCH_CAUSALPFN_CAP_NUM_NEIGHBOURS=0` | Exact upstream CausalPFN neighbor behavior, not this repo's safer default |
| `CAUSAL_BENCH_CAUSALFM_QUERY_BATCH_SIZE=1024` | CausalFM OOM fallback — lower further if it still OOMs |

Splitting foundation (GPU) and HPO (CPU) into separate passes matters in practice: FLAML's
search is entirely CPU-side, so an expensive GPU sits idle through it if run together.

Three known deviations from the reference baselines, so the results below aren't overstated:
- **IPW never gets a PEHE score** — it only ever produces a single scalar ATE (`supports_cate = False`), not a per-unit CATE, so its PEHE column always reads "N/A". Not a bug.
- **"Debiased ML" here is generic EconML `DML`, not the paper's Forest DML** — its numbers are a reasonable doubly-robust baseline, not a direct reproduction of that specific paper baseline.
- **CausalPFN's neighbor cap** (avoiding FAISS's `-1` sentinel behavior when a treatment arm is smaller than `num_neighbours`) produced about a 0.1% deviation from exact upstream behavior in a spot check — set `CAUSAL_BENCH_CAUSALPFN_CAP_NUM_NEIGHBOURS=0` above for exact parity instead.

In [ ]:
hpo_time_budget = float(os.environ.get("CAUSAL_BENCH_HPO_TIME_BUDGET", "900"))
if not np.isfinite(hpo_time_budget) or hpo_time_budget <= 0:
    raise ValueError("CAUSAL_BENCH_HPO_TIME_BUDGET must be a positive number of seconds")
HPO_CONFIG = HPOConfig(time_budget=hpo_time_budget, cv=3, verbose=0, early_stop=True)

# (display name, base name, wrapper class, "foundation" or "hpo")
MODEL_REGISTRY = [
    ("CausalPFN (Foundation)", "CausalPFN", CausalPFNWrapper, "foundation"),
    ("Do-PFN (Foundation)", "Do-PFN", DoPFNWrapper, "foundation"),
    ("CausalFM (Foundation)", "CausalFM", CausalFMWrapper, "foundation"),
    ("S-learner", "S-learner", SLearnerWrapper, "hpo"),
    ("T-learner", "T-learner", TLearnerWrapper, "hpo"),
    ("X-learner", "X-learner", XLearnerWrapper, "hpo"),
    ("Debiased ML", "Debiased ML", DebiasedMLWrapper, "hpo"),
    ("IPW", "IPW", IPWWrapper, "hpo"),
    ("DR (Doubly Robust)", "DR (Doubly Robust)", DRWrapper, "hpo"),
]
MODEL_SPECS = [
    {"display": display, "base": base, "cls": cls, "kind": kind,
     "supports_cate": bool(getattr(cls, "supports_cate", True))}
    for display, base, cls, kind in MODEL_REGISTRY
]

run_foundation = env_flag("CAUSAL_BENCH_RUN_FOUNDATION", True)
run_hpo = env_flag("CAUSAL_BENCH_RUN_HPO", True)
filter_tokens = {
    token.strip().casefold()
    for token in os.environ.get("CAUSAL_BENCH_MODEL_FILTER", "").split(",")
    if token.strip()
}
known_names = {name.casefold() for spec in MODEL_SPECS for name in (spec["display"], spec["base"])}
if filter_tokens - known_names:
    raise ValueError(f"Unknown model(s) in CAUSAL_BENCH_MODEL_FILTER: {sorted(filter_tokens - known_names)}")

SELECTED_SPECS = [
    spec for spec in MODEL_SPECS
    if ((spec["kind"] == "foundation" and run_foundation) or (spec["kind"] == "hpo" and run_hpo))
    and (not filter_tokens or spec["display"].casefold() in filter_tokens or spec["base"].casefold() in filter_tokens)
]
if not SELECTED_SPECS:
    raise RuntimeError("No models selected -- check CAUSAL_BENCH_RUN_FOUNDATION/RUN_HPO/MODEL_FILTER.")

def is_model_available(spec):
    if spec["base"] == "Do-PFN":
        return spec["cls"].is_available(repo_dir=str(DOPFN_DIR))
    if spec["base"] == "CausalFM":
        return spec["cls"].is_available(checkpoint_path=str(CAUSALFM_CHECKPOINT))
    return spec["cls"].is_available()

unavailable = [spec["display"] for spec in SELECTED_SPECS if not is_model_available(spec)]
if unavailable:
    raise RuntimeError(f"Selected models are unavailable in this environment (never silently skipped): {unavailable}")

if any(spec["kind"] == "hpo" for spec in SELECTED_SPECS):
    print(f"HPO is CPU-side; production default is 420 FLAML searches / ~105 budget-hours.")
print("Selected models:", [spec["display"] for spec in SELECTED_SPECS])

## 4. Load RealCause realizations and write a resume manifest

Default `N=10`: CausalPFN's arXiv v1 protocol. Each realization is an *independent*
generator draw — covariates and treatment assignment are not fixed across realizations, only
the real Lalonde data distribution they're drawn from. Per realization: `default_rng(42 + i)`
splits 90% train / 10% held-out test; PEHE is scored on the test split against its true ITE,
and ATE is scored from a *separate* full-data fit against `mean(ite)` — matching CausalPFN's
own `benchmarks/realcause.py` exactly (see `docs/LALONDE_DATASET.md`).

The manifest records exactly what produced this output — config, dependency/vendor pins, a
hash of every input CSV, and a hash of every `causal_bench` source file involved — so a
resumed run refuses to silently mix results from two different configurations.

In [ ]:
from causal_bench import ate_abs_error, ate_rel_error, evaluate_cate, load_lalonde_realcause
import causal_bench.data_loader as realcause_loader

EXPECTED_REALCAUSE_REVISION = "7fae8e26e4e584c99723aaf719f3b9627b369de7"
if realcause_loader._REALCAUSE_REVISION != EXPECTED_REALCAUSE_REVISION:
    raise RuntimeError(
        f"RealCause data revision mismatch: causal_bench uses "
        f"{realcause_loader._REALCAUSE_REVISION}, this notebook expects {EXPECTED_REALCAUSE_REVISION}"
    )

N_REALIZATIONS = int(os.environ.get("CAUSAL_BENCH_N_REALIZATIONS", "10"))
if not 1 <= N_REALIZATIONS <= 100:
    raise ValueError("CAUSAL_BENCH_N_REALIZATIONS must be between 1 and 100")
COHORTS = ("cps", "psid")  # CPS first: the larger cohort, a good early stress test
SPLIT_SEED = 42

def sha256_of(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def atomic_write_json(path, obj):
    tmp = path.with_name(f".{path.name}.{os.getpid()}.tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True, allow_nan=False)
        f.write("\n")
    os.replace(tmp, path)

def validate_realization(cohort, i, r):
    arrays = (r.X_train, r.T_train, r.Y_train, r.X_test, r.tau_true_test, r.X_full, r.T_full, r.Y_full)
    if r.realization != i or not all(np.isfinite(np.asarray(a)).all() for a in arrays):
        raise RuntimeError(f"{cohort} realization {i}: invalid (non-finite or mislabeled) data")
    if not np.isfinite(r.ate_true):
        raise RuntimeError(f"{cohort} realization {i}: non-finite ate_true")
    if any(np.asarray(x).ndim != 2 for x in (r.X_train, r.X_test, r.X_full)):
        raise RuntimeError(f"{cohort} realization {i}: X must be 2D")
    if not (len(r.X_train) == len(r.T_train) == len(r.Y_train)):
        raise RuntimeError(f"{cohort} realization {i}: train length mismatch")
    if len(r.X_test) != len(r.tau_true_test):
        raise RuntimeError(f"{cohort} realization {i}: test length mismatch")
    if not (len(r.X_full) == len(r.T_full) == len(r.Y_full)):
        raise RuntimeError(f"{cohort} realization {i}: full-data length mismatch")
    if not (r.X_train.shape[1] == r.X_test.shape[1] == r.X_full.shape[1]):
        raise RuntimeError(f"{cohort} realization {i}: feature-count mismatch across splits")
    if set(np.unique(r.T_train)) != {0.0, 1.0} or set(np.unique(r.T_full)) != {0.0, 1.0}:
        raise RuntimeError(f"{cohort} realization {i}: treatment must be binary {{0, 1}}")

query_batch_label = "native" if CAUSALFM_QUERY_BATCH_SIZE is None else str(CAUSALFM_QUERY_BATCH_SIZE)
run_stem = (
    f"realcause-lalonde-first{N_REALIZATIONS}_hpo{hpo_time_budget:g}s_cv3_split{SPLIT_SEED}"
    f"_seed{BASE_SEED}_cpfncap{int(CAUSALPFN_CAP_NEIGHBOURS)}_cfmbatch{query_batch_label}"
)
output_dir = Path(os.environ.get("CAUSAL_BENCH_OUTPUT_DIR", str(PROJECT_ROOT / "notebooks/results"))).expanduser().resolve() / run_stem
checkpoint_dir = output_dir / "checkpoints"
failure_dir = output_dir / "failures"
results_csv_path = output_dir / f"{run_stem}.csv"
manifest_path = output_dir / "manifest.json"
for path in (output_dir, checkpoint_dir, failure_dir):
    path.mkdir(parents=True, exist_ok=True)

realizations_by_cohort = {}
input_csv_sha256 = {}
for cohort in COHORTS:
    realizations = load_lalonde_realcause(cohort, n_realizations=N_REALIZATIONS, seed=SPLIT_SEED, test_ratio=0.1)
    if len(realizations) != N_REALIZATIONS:
        raise RuntimeError(f"{cohort}: expected {N_REALIZATIONS} realizations, got {len(realizations)}")
    for i, r in enumerate(realizations):
        validate_realization(cohort, i, r)
        cache_path = Path(realcause_loader._CACHE_DIR) / f"realcause_lalonde_{cohort}_sample{i}.csv"
        if not cache_path.is_file():
            raise RuntimeError(f"Expected cached RealCause input CSV is missing: {cache_path}")
        input_csv_sha256[cache_path.name] = sha256_of(cache_path)
    realizations_by_cohort[cohort] = realizations
    print(cohort, len(realizations), realizations[0].meta)

expected_hash_count = len(COHORTS) * N_REALIZATIONS
if len(input_csv_sha256) != expected_hash_count:
    raise RuntimeError(f"Expected {expected_hash_count} RealCause input hashes, got {len(input_csv_sha256)}")

causal_bench_source_files = [
    PROJECT_ROOT / "causal_bench" / name
    for name in ("wrap_foundation.py", "wrap_metalearners.py", "wrap_causalpfn.py",
                 "wrap_dopfn.py", "wrap_causalfm.py", "data_loader.py", "metrics.py")
]
run_config = {
    "schema": 3, "n": N_REALIZATIONS, "cohorts": list(COHORTS),
    "split_seed": SPLIT_SEED, "base_seed": BASE_SEED,
    "hpo": {"time_budget": hpo_time_budget, "cv": 3},
    "models": [spec["display"] for spec in MODEL_SPECS],
    "foundation": {"causalpfn_cap_num_neighbours": CAUSALPFN_CAP_NEIGHBOURS,
                   "causalfm_query_batch_size": CAUSALFM_QUERY_BATCH_SIZE},
    "versions": {dist: importlib_metadata.version(dist) for dist in PINNED_VERSIONS},
    "vendor_shas": VENDOR_SHAS,
    "source_sha256": {str(p.relative_to(PROJECT_ROOT)): sha256_of(p) for p in causal_bench_source_files},
    "realcause_data": {"revision": EXPECTED_REALCAUSE_REVISION,
                       "csv_sha256": dict(sorted(input_csv_sha256.items()))},
}
if manifest_path.exists():
    with manifest_path.open(encoding="utf-8") as f:
        existing_manifest = json.load(f)
    if existing_manifest.get("config") != run_config:
        raise RuntimeError(f"This run's config doesn't match the existing resume manifest: {manifest_path}")
else:
    atomic_write_json(manifest_path, {"created_utc": datetime.now(timezone.utc).isoformat(), "config": run_config})

print("RealCause input revision:", EXPECTED_REALCAUSE_REVISION)
print("Hashed cached RealCause CSVs:", len(input_csv_sha256))
print("Output directory:", output_dir)

## 5. Run every (model, cohort, realization) task — checkpointed and resumable

Each of the (up to 9 × 2 × 10 =) 180 tasks is checkpointed to its own atomic JSON file the
moment it completes; rerunning this cell skips anything already checkpointed, so an
interrupted run picks back up where it left off rather than restarting the full 105 hours.
Any failure is recorded (with a full traceback) to `failures/` and, since `FAIL_FAST=True`
by default, re-raised — a silently-skipped model would be far worse than a loud crash for a
benchmark whose whole point is completeness.

In [ ]:
def slugify(text):
    return re.sub(r"[^a-z0-9]+", "-", text.casefold()).strip("-")

def task_key(model_display, cohort, realization):
    return model_display, cohort, int(realization)

def checkpoint_path_for(task):
    model_display, cohort, realization = task
    return checkpoint_dir / f"{slugify(model_display)}__{cohort}__r{realization:02d}.json"

def task_seed(model_idx, cohort_idx, realization, stage):
    return BASE_SEED + model_idx * 100_000 + cohort_idx * 10_000 + int(realization) * 10 + (1 if stage == "cate" else 2)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def free_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def best_params_json(model):
    return json.dumps(getattr(model, "best_params_", {}) or {}, sort_keys=True, default=repr)

def as_validated_vector(x, expected_length, label):
    x = np.asarray(x, dtype=float).reshape(-1)
    if len(x) != expected_length or not np.isfinite(x).all():
        raise RuntimeError(f"Invalid {label}: shape={x.shape}")
    return x

def build_model(spec):
    if spec["base"] == "CausalPFN":
        return spec["cls"](device=device, cap_num_neighbours=CAUSALPFN_CAP_NEIGHBOURS)
    if spec["base"] == "CausalFM":
        return spec["cls"](checkpoint_path=str(CAUSALFM_CHECKPOINT), device=device,
                            query_batch_size=CAUSALFM_QUERY_BATCH_SIZE)
    if spec["base"] == "Do-PFN":
        return spec["cls"](repo_dir=str(DOPFN_DIR), device=device)
    if spec["kind"] == "foundation":
        return spec["cls"](device=device)
    return spec["cls"](device="cpu", hpo=True, hpo_config=HPO_CONFIG)  # metalearner: FLAML runs on CPU

ALL_TASK_KEYS = {task_key(spec["display"], cohort, i)
                  for spec in MODEL_SPECS for cohort in COHORTS for i in range(N_REALIZATIONS)}
SELECTED_TASK_KEYS = {task_key(spec["display"], cohort, i)
                       for spec in SELECTED_SPECS for cohort in COHORTS for i in range(N_REALIZATIONS)}

def read_checkpointed_records():
    records = {}
    for path in sorted(checkpoint_dir.glob("*.json")):
        with path.open(encoding="utf-8") as f:
            row = json.load(f)
        task = task_key(row["model"], row["cohort"], row["realization"])
        if task not in ALL_TASK_KEYS or task in records:
            raise RuntimeError(f"Unexpected or duplicate checkpoint file: {path}")
        records[task] = row
    return records

def rebuild_results_csv(records):
    frame = pd.DataFrame(sorted(records.values(), key=lambda row: (row["model"], row["cohort"], row["realization"])))
    tmp_path = results_csv_path.with_name(f".{results_csv_path.name}.{os.getpid()}.tmp")
    frame.to_csv(tmp_path, index=False)
    os.replace(tmp_path, results_csv_path)
    return frame

def record_failure(spec, cohort, realization, stage, seed, exc):
    now = datetime.now(timezone.utc)
    filename = f"{now.strftime('%Y%m%dT%H%M%S.%fZ')}__{os.getpid()}__{slugify(spec['display'])}__{cohort}__r{realization:02d}.json"
    atomic_write_json(failure_dir / filename, {
        "timestamp_utc": now.isoformat(), "model": spec["display"], "cohort": cohort,
        "realization": realization, "stage": stage, "seed": seed,
        "exception_type": type(exc).__name__, "exception": str(exc),
        "traceback": traceback.format_exc(),
    })

completed_records = read_checkpointed_records()
print(f"Resuming: {len(completed_records)}/{len(ALL_TASK_KEYS)} total tasks checkpointed; "
      f"{len(set(completed_records) & SELECTED_TASK_KEYS)}/{len(SELECTED_TASK_KEYS)} of the selected ones done.")

In [ ]:
for model_idx, spec in enumerate(MODEL_SPECS):
    if spec not in SELECTED_SPECS:
        continue

    for cohort_idx, cohort in enumerate(COHORTS):
        for r in realizations_by_cohort[cohort]:
            task = task_key(spec["display"], cohort, r.realization)
            if task in completed_records:
                continue

            stage, active_seed = "init", None
            try:
                tau = lower = upper = cate_runtime = cate_params = None
                if spec["supports_cate"]:
                    stage = "cate"
                    active_seed = task_seed(model_idx, cohort_idx, r.realization, stage)
                    seed_everything(active_seed)

                    start = time.perf_counter()
                    cate_model = build_model(spec)
                    cate_model.fit(r.X_train, r.T_train, r.Y_train)
                    prediction = cate_model.predict(r.X_test)
                    if not isinstance(prediction, tuple) or len(prediction) != 3:
                        raise RuntimeError("predict() must return a (tau, lower, upper) 3-tuple")
                    tau, lower, upper = prediction
                    tau = as_validated_vector(tau, len(r.X_test), "CATE prediction")
                    if (lower is None) != (upper is None):
                        raise RuntimeError("predict() returned only one of lower/upper")
                    if lower is not None:
                        lower = as_validated_vector(lower, len(tau), "lower interval bound")
                        upper = as_validated_vector(upper, len(tau), "upper interval bound")
                        if np.any(lower > upper):
                            raise RuntimeError("predicted lower bound exceeds upper bound")
                    cate_runtime = time.perf_counter() - start
                    cate_params = best_params_json(cate_model)
                    cate_model = None
                    free_gpu_memory()

                stage = "ate"
                active_seed = task_seed(model_idx, cohort_idx, r.realization, stage)
                seed_everything(active_seed)

                start = time.perf_counter()
                ate_model = build_model(spec)
                ate_model.fit(r.X_full, r.T_full, r.Y_full)
                if hasattr(ate_model, "estimate_ate"):
                    ate_hat = ate_model.estimate_ate(r.X_full, r.T_full, r.Y_full)
                else:
                    full_tau, _, _ = ate_model.predict(r.X_full)
                    ate_hat = float(as_validated_vector(full_tau, len(r.X_full), "full-data CATE").mean())
                ate_hat = np.asarray(ate_hat)
                if ate_hat.size != 1:
                    raise RuntimeError(f"ATE estimate must be scalar; got shape {ate_hat.shape}")
                ate_hat = float(ate_hat.reshape(-1)[0])
                if not np.isfinite(ate_hat):
                    raise RuntimeError("ATE estimate is non-finite")
                ate_runtime = time.perf_counter() - start
                ate_params = best_params_json(ate_model)
                ate_model = None
                free_gpu_memory()

                if spec["supports_cate"]:
                    metrics = evaluate_cate(tau, r.tau_true_test, ate_hat=ate_hat, ate_true=r.ate_true,
                                             lower=lower, upper=upper, runtime_s=cate_runtime + ate_runtime)
                else:
                    metrics = {
                        "pehe": None, "ate_hat": ate_hat, "ate_true": float(r.ate_true),
                        "ate_abs_error": ate_abs_error(ate_hat, r.ate_true),
                        "ate_rel_error": ate_rel_error(ate_hat, r.ate_true),
                        "bias": None, "coverage_95": None, "runtime_s": ate_runtime,
                    }

                row = {
                    **metrics,
                    "model": spec["display"], "base_model": spec["base"], "model_kind": spec["kind"],
                    "supports_cate": spec["supports_cate"], "cohort": cohort, "realization": int(r.realization),
                    "execution_device": device if spec["kind"] == "foundation" else "cpu",
                    "cate_runtime_s": cate_runtime, "ate_runtime_s": ate_runtime,
                    "cate_best_params": cate_params, "ate_best_params": ate_params,
                    "cate_seed": task_seed(model_idx, cohort_idx, r.realization, "cate") if spec["supports_cate"] else None,
                    "ate_seed": task_seed(model_idx, cohort_idx, r.realization, "ate"),
                    "hpo_time_budget_s": hpo_time_budget if spec["kind"] == "hpo" else None,
                    "hpo_cv": 3 if spec["kind"] == "hpo" else None,
                    "split_seed": SPLIT_SEED, "base_seed": BASE_SEED,
                }
                atomic_write_json(checkpoint_path_for(task), row)
                completed_records[task] = row
                rebuild_results_csv(completed_records)
                print(spec["display"], cohort, r.realization, "OK")

            except Exception as exc:
                record_failure(spec, cohort, r.realization, stage, active_seed, exc)
                print(spec["display"], cohort, r.realization, "FAILED", stage, repr(exc))
                if FAIL_FAST:
                    raise
            finally:
                cate_model = ate_model = None
                free_gpu_memory()

    clear_model_cache = getattr(spec["cls"], "clear_model_cache", None)
    if clear_model_cache is not None:
        removed = clear_model_cache()
        print(f"{spec['display']} cleared {removed} cached model(s)")
    free_gpu_memory()

results_frame = rebuild_results_csv(completed_records)
print(f"Checkpointed {len(completed_records)}/{len(ALL_TASK_KEYS)} total tasks.")

## 6. Results table

PEHE is displayed divided by 1,000, to compare directly against the paper's Table 1 (§0);
the raw CSV keeps original dollar units. "Completed" counts are scoped to models selected
*this* invocation, so a staged GPU-then-CPU run shows accurate progress at each stage.

In [ ]:
completed_records = read_checkpointed_records()
results_frame = rebuild_results_csv(completed_records)
selected_display_names = {spec["display"] for spec in SELECTED_SPECS}

summary_rows = []
for spec in MODEL_SPECS:
    for cohort in COHORTS:
        subset = (results_frame[(results_frame["model"] == spec["display"]) & (results_frame["cohort"] == cohort)]
                  if not results_frame.empty else pd.DataFrame())
        n_done = len(subset)

        if spec["supports_cate"] and n_done:
            pehe_thousands = pd.to_numeric(subset["pehe"], errors="coerce").dropna() / 1000
            pehe_display = (f"{pehe_thousands.mean():.2f} ± {pehe_thousands.sem():.2f}"
                             if len(pehe_thousands) > 1 else f"{pehe_thousands.mean():.2f} ± N/A")
        elif spec["supports_cate"]:
            pehe_display = "pending"
        else:
            pehe_display = "N/A"  # IPW has no CATE estimate, only ATE

        if n_done:
            ate_rel = pd.to_numeric(subset["ate_rel_error"], errors="coerce").dropna()
            ate_display = f"{ate_rel.mean():.2f} ± {ate_rel.sem():.2f}" if len(ate_rel) > 1 else f"{ate_rel.mean():.2f} ± N/A"
        else:
            ate_display = "pending"

        is_selected = spec["display"] in selected_display_names
        status = ("complete" if n_done == N_REALIZATIONS else "INCOMPLETE") if is_selected else "not selected"
        summary_rows.append({
            "model": spec["display"], "cohort": cohort.upper(),
            "PEHE (×10³), mean ± SEM": pehe_display, "ATE relative error, mean ± SEM": ate_display,
            "completed": f"{n_done}/{N_REALIZATIONS}", "status": status,
        })

summary_frame = pd.DataFrame(summary_rows)
print("RealCause Lalonde", "arXiv v1 first-10" if N_REALIZATIONS == 10 else f"first-{N_REALIZATIONS} override")
print(summary_frame.to_string(index=False))
print(f"\nRaw CSV: {results_csv_path}\nCheckpoints: {checkpoint_dir}")
print(f"Failures: {failure_dir} ({len(list(failure_dir.glob('*.json')))})\nManifest: {manifest_path}")

missing = sorted(SELECTED_TASK_KEYS - set(completed_records))
if missing:
    raise RuntimeError(f"Incomplete after saving: {len(missing)}/{len(SELECTED_TASK_KEYS)} selected tasks missing; first={missing[:10]}")
print(f"SUCCESS: all {len(SELECTED_TASK_KEYS)} selected tasks complete.")